# 📜 Nyaya-Jyoti: AI-Powered Will Document Generator

Welcome to **Nyaya-Jyoti**, an AI-powered assistant to help you create a legally formatted **Last Will and Testament** in a simple, conversational way.

This tool uses **GPT-Neo** along with **semantic matching** to generate personalized clauses — including special instructions for property, guardianship, or legacy items — based on your inputs.

---

### 📝 How it works:

1. **Upload the clause CSV file** and the **Will template (.docx)** with standard placeholders
2. **Answer easy questions** about the testator, family members, property, and witnesses
3. **Optionally specify a special bequest or instruction** (e.g., “gift gold ring to daughter”) — AI will generate the legal clause
4. **Watch the document update line-by-line**
5. **Download the complete Will** in `.docx` format

---

### ⚠️ Instructions:

- Upload the following:
  - ✅ Clause CSV file with prompts and examples
  - ✅ Word `.docx` template containing placeholders like `[Full_Name]`, `[Flat_Number]`, etc.
- Provide accurate responses (names, dates, and property details)
- The AI-generated special clause is inserted automatically at the correct place
- Document downloads automatically upon completion

---

> 🛡️ This Will generator is part of the *Nyaya-Jyoti* legal AI project at Bennett University and is intended for demonstration purposes.


In [ ]:
# @title
# 🛠️ Install required packages
!pip install -q transformers torch pandas sentence-transformers python-docx

# 📦 Imports
import torch
import pandas as pd
import re
import docx
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
from google.colab import files
from IPython.display import display, Markdown

# --- Load Clause Prompt Registry ---
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
prompt_df = pd.read_csv(csv_filename)
prompt_df["parameters"] = prompt_df["parameters"].fillna("").astype(str)
prompt_df["parameters"] = prompt_df["parameters"].apply(
    lambda x: ", ".join(sorted(set(p.strip() for p in x.split(",") if p.strip().lower() != "nan")))
)

# --- Load GPT-Neo & Embedding Model ---
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
instruction_embeddings = embedder.encode(prompt_df['instruction'].tolist(), convert_to_tensor=True)

# --- Clause Generation Utilities ---
def build_prompt(example_1, example_2, instruction):
    return (
        "You are a legal assistant specialized in drafting formal legal clauses.\n"
        f"Example 1:\nClause: {example_1}\nEndClause\n\n"
        f"Example 2:\nClause: {example_2}\nEndClause\n\n"
        f"Now, generate ONLY the legal clause for {instruction}, using formal legal language.\n"
        "Output only the text between the markers 'Clause:' and 'EndClause'.\n\nClause: "
    )

def generate_clause(prompt_text):
    input_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    output_ids = model.generate(
        input_ids,
        max_length=300,
        temperature=0.35,
        top_k=50,
        top_p=0.85,
        repetition_penalty=1.2,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id
    )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "EndClause" in generated_text:
        return generated_text.split("Clause:")[1].split("EndClause")[0].strip()
    return generated_text.split("Clause:")[1].strip()

def fill_parameters_dynamic(clause_text, param_string):
    placeholders = set(re.findall(r"{(.*?)}", clause_text))
    defined_params = [p.strip() for p in str(param_string).split(',') if p.strip()]
    combined_params = sorted(placeholders.union(set(defined_params)))
    param_values = {}
    for param in combined_params:
        value = input(f"🧾 Please provide value for '{param}': ").strip()
        param_values[param] = value
    for param, value in param_values.items():
        clause_text = clause_text.replace(f"{{{param}}}", value)
    return clause_text, param_values

def find_best_match_semantic(user_input):
    user_embedding = embedder.encode(user_input, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(user_embedding, instruction_embeddings)[0]
    best_idx = torch.argmax(cosine_scores).item()
    best_score = cosine_scores[best_idx].item()

    if best_score > 0.4:
        return prompt_df.iloc[best_idx]

    return None  # fallback handled later

# --- Upload Template ---
print("📂 Upload your Will DOCX template:")
uploaded_template = files.upload()
template_filename = list(uploaded_template.keys())[0]
doc = docx.Document(template_filename)

# --- Placeholder Explanations ---
explanations = {
    "Full_Name": "Enter full name of the person making the will",
    "Father_Name": "Enter father's name of the testator",
    "Age": "Enter the age of the testator",
    "Full_Address": "Enter the full residential address of the testator",
    "Executor_Relationship": "Relationship of executor to testator (e.g., son, daughter)",
    "Executor_Name": "Full name of the Executor",
    "Executor_Address": "Address of the Executor",
    "Wife_Name": "Enter the name of the wife (if applicable)",
    "First_Child_Name": "Enter name of the first child (if any)",
    "Second_Child_Name": "Enter name of the second child (if any)",
    "Flat_Number": "Flat number owned by the testator",
    "Flat_Address": "Address of the flat owned",
    "Day": "Enter day of execution (e.g., 14)",
    "Month": "Enter month of execution (e.g., April)",
    "Year": "Enter year of execution (e.g., 2025)",
    "Place": "Enter the city or town where the Will is signed",
    "First_Witness_Name": "Name of first witness",
    "Second_Witness_Name": "Name of second witness",
    "Beneficiary_Name": "Person receiving specific property (for example clause)",
    "Property_Description": "Describe the specific property (for example clause)",
    "Guardian_Name": "Name of guardian for minor children",
    "Special_Clauses": "Special clause auto-generated by AI"
}

# --- Fill Placeholders First ---
print("\n📋 Please answer the following questions to populate the will:")
user_inputs = {}
for placeholder, explanation in explanations.items():
    if placeholder == "Special_Clauses":
        continue
    value = input(f"🖋 {explanation}: ").strip()
    user_inputs[placeholder] = value
    for para in doc.paragraphs:
        if f"[{placeholder}]" in para.text:
            original_text = para.text
            para.text = para.text.replace(f"[{placeholder}]", value)
            display(Markdown(f"**📄 Updated Line:**\n\n`Before:` {original_text}\n\n`After:` {para.text}"))

# --- Special Clause Prompt Comes AFTER Placeholder Filling ---
special_clause_text = ""
print("\n📄 All standard fields are now filled.")
special_query = input("\n💬 Do you have any special clause to include (e.g., 'leave remaining estate to son', 'gift jewelry to daughter')?\n📨 Special Request (leave blank if none): ").strip()

if special_query:
    match_row = find_best_match_semantic(special_query)
    if match_row is not None:
        prompt = build_prompt(match_row['example_1'], match_row['example_2'], match_row['instruction'])
        raw_clause = generate_clause(prompt)
        special_clause_text, _ = fill_parameters_dynamic(raw_clause, match_row.get('parameters', ''))
    else:
        # fallback direct generation
        special_clause_text = generate_clause(
            f"You are a legal assistant. Draft a will clause for the following instruction: {special_query}. Use formal language.\n\nClause: "
        )

# --- Insert Special Clause in Template ---
if special_clause_text:
    for para in doc.paragraphs:
        if "[Special_Clauses]" in para.text:
            para.text = para.text.replace("[Special_Clauses]", special_clause_text.strip())
            display(Markdown(f"**📄 Inserted Special Clause:** {para.text}"))
            break

# --- Save Final Will ---
output_file = "completed_will.docx"
doc.save(output_file)
files.download(output_file)
print(f"\n✅ Will document saved as: {output_file}")


Saving Updated_Will_Clauses.csv to Updated_Will_Clauses (4).csv
📂 Upload your Will DOCX template:


Saving simple_will_updated.docx to simple_will_updated (4).docx

📋 Please answer the following questions to populate the will:
🖋 Enter full name of the person making the will: s


**📄 Updated Line:**

`Before:` I, [Full_Name], son/daughter of Shri [Father_Name], aged [Age] years, resident of [Full_Address], do hereby revoke all my former Wills, Codicils, and Testamentary dispositions made by me. I declare this to be my last Will and Testament.

`After:` I, s, son/daughter of Shri [Father_Name], aged [Age] years, resident of [Full_Address], do hereby revoke all my former Wills, Codicils, and Testamentary dispositions made by me. I declare this to be my last Will and Testament.

🖋 Enter father's name of the testator: 


**📄 Updated Line:**

`Before:` I, s, son/daughter of Shri [Father_Name], aged [Age] years, resident of [Full_Address], do hereby revoke all my former Wills, Codicils, and Testamentary dispositions made by me. I declare this to be my last Will and Testament.

`After:` I, s, son/daughter of Shri , aged [Age] years, resident of [Full_Address], do hereby revoke all my former Wills, Codicils, and Testamentary dispositions made by me. I declare this to be my last Will and Testament.

...

🖋 Person receiving specific property (for example clause): 
🖋 Describe the specific property (for example clause): 
🖋 Name of guardian for minor children: 

📄 All standard fields are now filled.

💬 Do you have any special clause to include (e.g., 'leave remaining estate to son', 'gift jewelry to daughter')?
📨 Special Request (leave blank if none): Give my car to my daughter Riya.


**📄 Inserted Special Clause:** __________________________________________________

Riya: I want to give my car to my daughter Riya.

IN WITNESS WHEREOF I have hereunto set my hands on this  day of ,  at .

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Will document saved as: completed_will.docx
